# Forensic Sketch Generator — Cloud GPU Setup

**Portfolio / technical demo notebook.** This spins up the backend on a Colab GPU runtime and exposes it via ngrok so the local `frontend/index.html` dashboard can talk to it.

> This project is a technical demonstration only. Matches are computed against the bundled synthetic `mock_database` images (or random scores in CPU/no-GPU mock mode) — it is not a validated forensic tool and should not be used to make real identification decisions about real people.

Run the cells in order, top to bottom. Make sure **Runtime > Change runtime type > GPU (T4)** is selected before you start.

### 1. Clone the repo and install dependencies

In [ ]:
%cd /content
!rm -rf Forensic-Sketch-Generator

# Clone your repository
!git clone -q https://github.com/steve21-coding/Forensic-Sketch-Generator.git
%cd /content/Forensic-Sketch-Generator

# Cache path for InsightFace's own auto-downloaded model weights
!mkdir -p ~/.insightface/models

# Web server dependencies
!pip install -q fastapi "uvicorn[standard]" pyngrok nest-asyncio python-multipart

# Generative modeling stack
!pip install -U -q "bitsandbytes>=0.46.1" transformers accelerate diffusers safetensors

# Biometric / vector search stack
!pip install -U -q insightface onnxruntime-gpu faiss-cpu opencv-python-headless gdown

# Semantic face-parsing model (BiSeNet) + pretrained checkpoint
!git clone -q https://github.com/zllrunning/face-parsing.PyTorch.git
!mkdir -p face-parsing.PyTorch/res/cp
!wget -q -O face-parsing.PyTorch/res/cp/79999_iter.pth https://huggingface.co/manyotherfunctions/face-parse-bisent/resolve/main/79999_iter.pth

print('\u2713 Environment setup complete.')

### 2. Verify the backend package imports cleanly

This is a smoke test only — it just confirms `backend/` (schemas, config, errors, services, main) imports without error *before* we try to boot the actual server. If this cell fails, fix the traceback here first; a broken import at this stage is exactly what causes `ERR_NGROK_8012` later (the server process dies on boot and nothing is ever listening on port 8000).

In [ ]:
import sys
sys.path.insert(0, '/content/Forensic-Sketch-Generator')

LOCAL_DB_PATH = '/content/Forensic-Sketch-Generator/mock_database'
print(f'Sample/demo database path: {LOCAL_DB_PATH}')

# [Optional] Uncomment to source your own demo images from Google Drive instead
# from google.colab import drive
# drive.mount('/content/drive')
# DRIVE_DB_PATH = '/content/drive/MyDrive/your_demo_images_folder'
# LOCAL_DB_PATH = '/content/local_demo_images'
# import shutil, os
# if os.path.exists(DRIVE_DB_PATH):
#     if os.path.exists(LOCAL_DB_PATH): shutil.rmtree(LOCAL_DB_PATH)
#     shutil.copytree(DRIVE_DB_PATH, LOCAL_DB_PATH)

try:
    from backend.config import models_singleton
    from backend.main import app
    print('\u2713 backend package imports cleanly. Ready to boot the server.')
except Exception as e:
    print(f'\u2717 Import failed: {type(e).__name__}: {e}')
    raise

### 3. ngrok token

Store your token as a Colab secret instead of pasting it in plaintext (click the key icon in the left sidebar, add `NGROK_AUTHTOKEN`, and toggle notebook access on). This keeps it out of your notebook file and safe if you ever commit this notebook to GitHub.

If you'd rather paste it directly for a quick one-off run, replace the line below with `NGROK_AUTHTOKEN = "your_token_here"` — just remember to remove it again before sharing or committing the notebook.

In [ ]:
from google.colab import userdata
NGROK_AUTHTOKEN = userdata.get('NGROK_AUTHTOKEN')
print('\u2713 ngrok token loaded from Colab secrets.')

### 4. Boot the server and open the ngrok tunnel

In [ ]:
import nest_asyncio
import uvicorn
import threading
from pyngrok import ngrok, conf

nest_asyncio.apply()
conf.get_default().auth_token = NGROK_AUTHTOKEN

# Close any lingering tunnels from a previous run in this session
ngrok.kill()

tunnel = ngrok.connect(8000, 'http')
public_url = tunnel.public_url

print('\n' + '='*70)
print(' Public URL:')
print(f' {public_url}')
print('='*70)
print(' Paste this URL into the "ngrok API URL" field in index.html and click CONNECT.')
print('='*70 + '\n')

def run_server():
    uvicorn.run('backend.main:app', host='0.0.0.0', port=8000, log_level='info')

t = threading.Thread(target=run_server, daemon=True)
t.start()

print('Server thread started. Model loading begins now (SDXL + InsightFace + BiSeNet on GPU, or Mock Mode without one).')
print('Continue to the next cell — it will wait for the server to come up and report status automatically.')

### 5. Wait for the server to come up (auto-retrying health check)

Model loading (especially SDXL) can take one to a few minutes on first boot. This cell polls `/health` every 5 seconds for up to 5 minutes instead of failing after a single attempt, and it also tells you plainly if the server thread has already died so you're not left staring at `ERR_NGROK_8012` with no explanation.

In [ ]:
import time, json, requests

MAX_WAIT_SECONDS = 300
POLL_EVERY = 5

waited = 0
up = False

while waited < MAX_WAIT_SECONDS:
    if not t.is_alive():
        print('\u2717 Server thread has died. Scroll up through the output of the previous cell for the traceback —')
        print('  that will show the real error (e.g. a bad import, missing checkpoint, or CUDA OOM).')
        break
    try:
        r = requests.get(f'{public_url}/health', timeout=3)
        print('\u2713 Server is up:')
        print(json.dumps(r.json(), indent=2))
        up = True
        break
    except Exception as e:
        print(f'[{waited}s] not up yet ({type(e).__name__}) — still waiting...')
        time.sleep(POLL_EVERY)
        waited += POLL_EVERY

if not up and t.is_alive():
    print(f'\u2717 Server did not respond within {MAX_WAIT_SECONDS}s. Re-run this cell to keep waiting,')
    print('  or check the Cell 4 output above for errors.')

### 6. (Optional) Build the demo index directly from this notebook

You can also do this from the dashboard UI once connected — this cell is just a quick way to confirm indexing works end-to-end without leaving Colab.

In [ ]:
import requests

resp = requests.post(f'{public_url}/api/build_index', json={'folder_path': LOCAL_DB_PATH})
print(resp.status_code)
print(resp.json())